# Prepare the latest saved model for deployment
Click Runtime → Run all and authorize Drive. This makes a weights-only copy of the latest complete checkpoint; it does not stop training or deploy the model. Tell your assistant when it says Ready.


In [ ]:
from google.colab import drive
from pathlib import Path
import hashlib
import torch

drive.mount('/content/drive')
folder = Path('/content/drive/MyDrive/Sanskrit-LLM/colab-1b')
checkpoints = sorted(p for p in folder.glob('tokens-*.pt')
                     if Path(str(p) + '.sha256').is_file())
if not checkpoints:
    raise RuntimeError('No complete checkpoint found in ' + str(folder))
checkpoint = checkpoints[-1]
h = hashlib.sha256()
with checkpoint.open('rb') as f:
    for block in iter(lambda: f.read(1024 * 1024), b''):
        h.update(block)
expected = Path(str(checkpoint) + '.sha256').read_text().split()[0]
if h.hexdigest() != expected:
    raise RuntimeError('Checkpoint checksum mismatch: ' + checkpoint.name)
data = torch.load(checkpoint, map_location='cpu', weights_only=False)
out = folder.parent / 'deployment-weights.pt'
tmp = folder.parent / 'deployment-weights.pt.tmp'
torch.save({'config': data['config'], 'state_dict': data['state_dict'],
            'metadata': data['metadata']}, tmp)
tmp.replace(out)
print('Ready:', out, 'from', checkpoint.name, flush=True)
print('Additional positions:', data['metadata']['additional_nonpadding_positions'], flush=True)
